# Step 10 — Evidence-Grounded Career Recommendation System

This notebook generates personalized career-development recommendations using:

- Competence scores
- The selected Competence → Employability prediction model
- SHAP-based prediction explanations
- Deterministic strength and improvement-area identification
- OpenAI API for natural-language recommendation generation
- A verified learning-resource library
- Rule-based fallback recommendations

The language model does not calculate or modify the Employability prediction. It only converts validated model evidence into clear, practical, and personalized guidance.

In [173]:
from pathlib import Path

import os
import json
import numpy as np
import pandas as pd
import shap

from dotenv import load_dotenv
from openai import OpenAI

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [174]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "phase2_competence":
    PROJECT_DIR = PROJECT_DIR.parent.parent

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "phase2_competence_prepared.csv"
)

RESULT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase2"
    / "recommendation_system"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project directory:", PROJECT_DIR)
print("Dataset:", DATA_PATH)
print("Results:", RESULT_DIR)

Project directory: d:\internship-employability-project
Dataset: d:\internship-employability-project\data\processed\phase2_competence_prepared.csv
Results: d:\internship-employability-project\reports\phase2\recommendation_system


In [175]:
load_dotenv(
    PROJECT_DIR / ".env"
)

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

print(
    "API key loaded:",
    OPENAI_API_KEY is not None
)

API key loaded: True


In [176]:
client = OpenAI(
    api_key=OPENAI_API_KEY
)

In [177]:
df = pd.read_csv(
    DATA_PATH
)

print(
    "Dataset shape:",
    df.shape
)

df.head()

Dataset shape: (1000, 34)


,Response_ID,C1,C2,C3,C4,C5,C6,C7,C8,C9,...,Comp_avg,EVS_avg,LifeCareer_avg,LearningInnovation_avg,ITMedia_avg,HumanCapital_avg,SocialCapital_avg,CareerIdentity_avg,EnvironmentalMonitoring_avg,Balance_avg
0,1,3,3,4,4,3,3,3,3,4,...,3.3,4.750000,3.50,3.000000,3.333333,5.000000,4.0,4.75,5.0,5.0
1,2,3,4,4,3,4,4,4,4,4,...,3.7,3.500000,3.50,4.000000,3.666667,3.666667,3.5,3.75,4.0,2.5
2,3,3,4,3,3,3,4,3,4,4,...,3.4,3.750000,3.25,3.333333,3.666667,3.666667,4.0,3.75,4.0,3.5
3,4,4,4,4,4,4,4,4,4,4,...,3.9,4.000000,4.00,4.000000,3.666667,4.000000,4.0,4.00,4.0,4.0
4,5,4,4,4,4,4,5,5,4,4,...,4.3,4.333333,4.00,4.666667,4.333333,4.000000,4.5,4.50,5.0,4.0


In [178]:
FEATURES = [
    "LifeCareer_avg",
    "LearningInnovation_avg",
    "ITMedia_avg"
]

TARGET = "EVS_avg"

FEATURE_LABELS = {
    "LifeCareer_avg": "Life & Career Skills",
    "LearningInnovation_avg": "Learning & Innovation Skills",
    "ITMedia_avg": "IT & Media Skills"
}

X = df[FEATURES].copy()
y = df[TARGET].copy()

In [179]:
ITEM_LABELS = {
    "C1": "Adaptability to changing situations",
    "C2": "Communication with diverse team members",
    "C3": "Project planning and execution",
    "C4": "Leadership ability",

    "C5": "Reasoning and decision-making",
    "C6": "Communication and collaboration",
    "C7": "Creativity and innovation",

    "C8": "Information access and evaluation",
    "C9": "Media communication skills",
    "C10": "Media analysis and creation"
}

AREA_ITEMS = {
    "LifeCareer_avg": [
        "C1",
        "C2",
        "C3",
        "C4"
    ],

    "LearningInnovation_avg": [
        "C5",
        "C6",
        "C7"
    ],

    "ITMedia_avg": [
        "C8",
        "C9",
        "C10"
    ]
}

In [180]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(
    "Training samples:",
    len(X_train)
)

print(
    "Testing samples:",
    len(X_test)
)

Training samples: 800
Testing samples: 200


In [181]:
selected_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LinearRegression()
    )
])

selected_model.fit(
    X_train,
    y_train
)

print(
    "Selected Linear Regression model ready."
)

Selected Linear Regression model ready.


SHAP Explainer

In [182]:
scaler = selected_model.named_steps[
    "scaler"
]

linear_model = selected_model.named_steps[
    "model"
]

X_train_scaled = pd.DataFrame(
    scaler.transform(
        X_train
    ),
    columns=FEATURES,
    index=X_train.index
)

explainer = shap.LinearExplainer(
    linear_model,
    X_train_scaled
)

print(
    "SHAP explainer ready."
)

Background dataset has 800 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=800 when initializing the masker.


SHAP explainer ready.


## Competence Areas Used for Recommendations

### Life & Career Skills
Includes adaptability, communication with diverse team members, project planning and execution, and leadership ability.

### Learning & Innovation Skills
Includes reasoning and decision-making, communication and collaboration, and creativity and innovation.

### IT & Media Skills
Includes information access and evaluation, media communication skills, and media analysis and creation.

Area Metadata

In [183]:
AREA_DETAILS = {
    "LifeCareer_avg": {
        "label": "Life & Career Skills",

        "components": [
            "Adaptability",
            "Communication with diverse team members",
            "Project planning and execution",
            "Leadership ability"
        ],

        "career_relevance": (
            "These skills support adaptability, teamwork, planning, "
            "leadership, and effective functioning in professional environments."
        )
    },

    "LearningInnovation_avg": {
        "label": "Learning & Innovation Skills",

        "components": [
            "Reasoning and decision-making",
            "Communication and collaboration",
            "Creativity and innovation"
        ],

        "career_relevance": (
            "These skills support problem-solving, collaboration, "
            "continuous learning, creativity, and innovation in the workplace."
        )
    },

    "ITMedia_avg": {
        "label": "IT & Media Skills",

        "components": [
            "Information access and evaluation",
            "Media communication skills",
            "Media analysis and creation"
        ],

        "career_relevance": (
            "These skills support digital communication, information literacy, "
            "professional media use, and technology-supported work."
        )
    }
}

Score interpretation

In [184]:
def interpret_feature_level(score):

    if score >= 4.0:
        return "Strong"

    elif score >= 3.0:
        return "Developing"

    else:
        return "Needs Improvement"

Learning Resource Library

In [185]:
LEARNING_RESOURCES = {

    "LifeCareer_avg": [
        {
            "provider": "OpenLearn",
            "resource_type": "Free Course / Learning Module",
            "topic": "Leadership and Management",
            "search_hint": "OpenLearn leadership management"
        },

        {
            "provider": "YouTube",
            "resource_type": "Video Learning",
            "topic": "Project Management Fundamentals",
            "search_hint": "project management fundamentals beginners"
        },

        {
            "provider": "YouTube",
            "resource_type": "Video Learning",
            "topic": "Professional Communication and Teamwork",
            "search_hint": "professional communication teamwork skills"
        }
    ],

    "LearningInnovation_avg": [
        {
            "provider": "Coursera",
            "resource_type": "Free-to-Audit Course Topic",
            "topic": "Creative Thinking and Problem Solving",
            "search_hint": "Coursera creative thinking problem solving"
        },

        {
            "provider": "YouTube",
            "resource_type": "Video Learning",
            "topic": "Design Thinking for Beginners",
            "search_hint": "design thinking for beginners"
        },

        {
            "provider": "YouTube",
            "resource_type": "Video Learning",
            "topic": "Critical Thinking and Decision Making",
            "search_hint": "critical thinking decision making"
        }
    ],

    "ITMedia_avg": [
        {
            "provider": "Microsoft Learn",
            "resource_type": "Free Learning Path",
            "topic": "Digital Skills and Information Literacy",
            "search_hint": "Microsoft Learn digital skills"
        },

        {
            "provider": "YouTube",
            "resource_type": "Video Learning",
            "topic": "Information and Media Literacy",
            "search_hint": "information media literacy"
        },

        {
            "provider": "YouTube",
            "resource_type": "Video Learning",
            "topic": "Professional Digital Communication",
            "search_hint": "professional digital communication"
        }
    ]
}

Build model evidence

In [186]:
def build_recommendation_evidence(
    input_row,
    model,
    explainer,
    scaler
):

    input_df = pd.DataFrame(
        [input_row],
        columns=FEATURES
    )

    predicted_evs = float(
        model.predict(
            input_df
        )[0]
    )

    scaled_input = pd.DataFrame(
        scaler.transform(
            input_df
        ),
        columns=FEATURES
    )

    shap_result = explainer(
        scaled_input
    )

    shap_values = (
        shap_result.values[0]
    )

    rows = []

    for feature, shap_value in zip(
        FEATURES,
        shap_values
    ):

        score = float(
            input_df.iloc[0][
                feature
            ]
        )

        rows.append({
            "feature": feature,
            "label": FEATURE_LABELS[
                feature
            ],
            "score": score,
            "level": interpret_feature_level(
                score
            ),
            "shap_value": float(
                shap_value
            ),
            "absolute_shap": abs(
                float(
                    shap_value
                )
            ),
            "effect": (
                "Increases Prediction"
                if shap_value >= 0
                else "Decreases Prediction"
            ),
            "career_relevance": (
                AREA_DETAILS[
                    feature
                ]["career_relevance"]
            )
        })

    evidence_df = pd.DataFrame(
        rows
    )

    strongest = (
        evidence_df
        .sort_values(
            "score",
            ascending=False
        )
        .iloc[0]
        .to_dict()
    )

    weakest = (
        evidence_df
        .sort_values(
            "score",
            ascending=True
        )
        .iloc[0]
        .to_dict()
    )

    impact_ranking = (
        evidence_df
        .sort_values(
            "absolute_shap",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    return {
        "predicted_evs_avg": predicted_evs,
        "feature_evidence": evidence_df,
        "strongest_feature": strongest,
        "weakest_feature": weakest,
        "impact_ranking": impact_ranking
    }

In [187]:
def build_item_evidence(
    survey_response
):

    rows = []

    for area_feature, items in (
        AREA_ITEMS.items()
    ):

        area_label = FEATURE_LABELS[
            area_feature
        ]

        for item in items:

            score = float(
                survey_response[
                    item
                ]
            )

            rows.append({
                "area": area_label,
                "item": item,
                "skill": ITEM_LABELS[
                    item
                ],
                "score": score,
                "level": (
                    interpret_feature_level(
                        score
                    )
                )
            })

    return pd.DataFrame(
        rows
    )

In [188]:
sample_survey = {

    "C1": 4,
    "C2": 4,
    "C3": 3,
    "C4": 4,

    "C5": 3,
    "C6": 4,
    "C7": 3,

    "C8": 4,
    "C9": 5,
    "C10": 4
}

Example Candidate

In [189]:
sample_input = {

    "LifeCareer_avg": np.mean([
        sample_survey["C1"],
        sample_survey["C2"],
        sample_survey["C3"],
        sample_survey["C4"]
    ]),

    "LearningInnovation_avg": np.mean([
        sample_survey["C5"],
        sample_survey["C6"],
        sample_survey["C7"]
    ]),

    "ITMedia_avg": np.mean([
        sample_survey["C8"],
        sample_survey["C9"],
        sample_survey["C10"]
    ])
}

sample_input

{'LifeCareer_avg': np.float64(3.75),
 'LearningInnovation_avg': np.float64(3.3333333333333335),
 'ITMedia_avg': np.float64(4.333333333333333)}

Select learning resources deterministically

In [190]:
evidence = (
    build_recommendation_evidence(
        sample_input,
        selected_model,
        explainer,
        scaler
    )
)

print(
    "Predicted EVS_avg:",
    evidence[
        "predicted_evs_avg"
    ]
)

evidence[
    "feature_evidence"
]

Predicted EVS_avg: 4.054798535255626


,feature,label,score,level,shap_value,absolute_shap,effect,career_relevance
0,LifeCareer_avg,Life & Career Skills,3.750000,Developing,-0.008809,0.008809,Decreases Prediction,"These skills support adaptability, teamwork, p..."
1,LearningInnovation_avg,Learning & Innovation Skills,3.333333,Developing,-0.146000,0.146000,Decreases Prediction,"These skills support problem-solving, collabor..."
2,ITMedia_avg,IT & Media Skills,4.333333,Strong,0.148778,0.148778,Increases Prediction,"These skills support digital communication, in..."


In [191]:
item_evidence = (
    build_item_evidence(
        sample_survey
    )
)

item_evidence

,area,item,skill,score,level
0,Life & Career Skills,C1,Adaptability to changing situations,4.0,Strong
1,Life & Career Skills,C2,Communication with diverse team members,4.0,Strong
2,Life & Career Skills,C3,Project planning and execution,3.0,Developing
3,Life & Career Skills,C4,Leadership ability,4.0,Strong
4,Learning & Innovation Skills,C5,Reasoning and decision-making,3.0,Developing
5,Learning & Innovation Skills,C6,Communication and collaboration,4.0,Strong
6,Learning & Innovation Skills,C7,Creativity and innovation,3.0,Developing
7,IT & Media Skills,C8,Information access and evaluation,4.0,Strong
8,IT & Media Skills,C9,Media communication skills,5.0,Strong
9,IT & Media Skills,C10,Media analysis and creation,4.0,Strong


In [192]:
def select_learning_resources(
    evidence,
    max_resources_per_area=2
):

    selected = []

    evidence_sorted = (
        evidence[
            "feature_evidence"
        ]
        .sort_values(
            "score",
            ascending=True
        )
    )

    for _, row in (
        evidence_sorted.iterrows()
    ):

        feature = row[
            "feature"
        ]

        if row["score"] < 4.0:

            for resource in (
                LEARNING_RESOURCES[
                    feature
                ][:max_resources_per_area]
            ):

                selected.append({
                    "area": (
                        FEATURE_LABELS[
                            feature
                        ]
                    ),
                    "provider": (
                        resource[
                            "provider"
                        ]
                    ),
                    "resource_type": (
                        resource[
                            "resource_type"
                        ]
                    ),
                    "topic": (
                        resource[
                            "topic"
                        ]
                    ),
                    "search_hint": (
                        resource[
                            "search_hint"
                        ]
                    )
                })

    return selected

In [193]:
selected_resources = (
    select_learning_resources(
        evidence
    )
)

selected_resources

[{'area': 'Learning & Innovation Skills',
  'provider': 'Coursera',
  'resource_type': 'Free-to-Audit Course Topic',
  'topic': 'Creative Thinking and Problem Solving',
  'search_hint': 'Coursera creative thinking problem solving'},
 {'area': 'Learning & Innovation Skills',
  'provider': 'YouTube',
  'resource_type': 'Video Learning',
  'topic': 'Design Thinking for Beginners',
  'search_hint': 'design thinking for beginners'},
 {'area': 'Life & Career Skills',
  'provider': 'OpenLearn',
  'resource_type': 'Free Course / Learning Module',
  'topic': 'Leadership and Management',
  'search_hint': 'OpenLearn leadership management'},
 {'area': 'Life & Career Skills',
  'provider': 'YouTube',
  'resource_type': 'Video Learning',
  'topic': 'Project Management Fundamentals',
  'search_hint': 'project management fundamentals beginners'}]

In [194]:
RULE_BASED_RECOMMENDATIONS = {

    "LifeCareer_avg": [
        "Take responsibility for planning a small academic or team project.",
        "Practice giving short presentations and leading group discussions.",
        "Use a weekly planning system to improve organization and adaptability."
    ],

    "LearningInnovation_avg": [
        "Practice solving one structured case study each week.",
        "Participate in group brainstorming or innovation activities.",
        "Use decision-making frameworks to compare alternative solutions."
    ],

    "ITMedia_avg": [
        "Practice evaluating the reliability of online information sources.",
        "Create professional presentations or digital content for academic projects.",
        "Improve digital communication through structured email, reports, and presentations."
    ]
}

In [195]:
def generate_rule_based_recommendation(
    evidence,
    item_evidence
):

    area_analysis = []

    for feature in FEATURES:

        area_label = (
            FEATURE_LABELS[
                feature
            ]
        )

        area_items = item_evidence[
            item_evidence["area"]
            == area_label
        ].copy()

        strengths_df = (
            area_items[
                area_items[
                    "score"
                ] >= 4.0
            ]
            .sort_values(
                "score",
                ascending=False
            )
        )

        improvement_df = (
            area_items[
                area_items[
                    "score"
                ] < 4.0
            ]
            .sort_values(
                "score",
                ascending=True
            )
        )

        strengths = (
            strengths_df[
                "skill"
            ].tolist()
        )

        improvements = (
            improvement_df[
                "skill"
            ].tolist()
        )

        if not strengths:
            strengths = [
                (
                    "This competence area "
                    "is still developing."
                )
            ]

        if not improvements:
            improvements = [
                (
                    "Maintain current performance "
                    "and continue advanced development."
                )
            ]

        area_analysis.append({

            "area": area_label,

            "strengths": strengths[
                :3
            ],

            "areas_for_improvement": (
                improvements[
                    :3
                ]
            ),

            "career_readiness_impact": (
                AREA_DETAILS[
                    feature
                ][
                    "career_relevance"
                ]
            ),

            "practical_suggestions": (
                RULE_BASED_RECOMMENDATIONS[
                    feature
                ][:3]
            )
        })

    top_strengths_df = (
        item_evidence
        .sort_values(
            "score",
            ascending=False
        )
        .head(3)
    )

    top_improvements_df = (
        item_evidence
        .sort_values(
            "score",
            ascending=True
        )
        .head(3)
    )

    selected_resources = (
        select_learning_resources(
            evidence
        )
    )

    formatted_resources = []

    for resource in (
        selected_resources
    ):

        formatted_resources.append({

            "area": resource[
                "area"
            ],

            "provider": resource[
                "provider"
            ],

            "resource_type": resource[
                "resource_type"
            ],

            "topic": resource[
                "topic"
            ],

            "reason": (
                "Recommended to support "
                f"development in "
                f"{resource['area']}."
            )
        })

    return {

        "area_analysis": (
            area_analysis
        ),

        "top_3_strengths": [
            (
                f"{row['skill']} "
                f"({row['score']:.1f}/5)"
            )
            for _, row in (
                top_strengths_df.iterrows()
            )
        ],

        "top_3_areas_for_improvement": [
            (
                f"{row['skill']} "
                f"({row['score']:.1f}/5)"
            )
            for _, row in (
                top_improvements_df.iterrows()
            )
        ],

        "personalized_action_plan": [
            (
                "Focus first on the three "
                "lowest-scoring competence items."
            ),
            (
                "Complete practical activities "
                "related to the priority area."
            ),
            (
                "Review progress after four weeks "
                "and repeat the competence assessment."
            )
        ],

        "overall_career_readiness_summary": (
            "The competence profile contains both "
            "stronger and developing areas. Career "
            "development should focus on improving "
            "lower-scoring competencies while "
            "maintaining current strengths."
        ),

        "recommended_next_steps": [
            (
                "Choose one priority competence "
                "area for immediate development."
            ),
            (
                "Complete at least one practical "
                "learning activity or project."
            ),
            (
                "Track improvement and reassess "
                "progress."
            )
        ],

        "recommended_learning_resources": (
            formatted_resources
        ),

        "source": (
            "rule_based_fallback"
        )
    }

In [196]:
def build_api_prompt(
    evidence,
    item_evidence,
    selected_resources
):

    area_blocks = []

    for _, row in evidence[
        "feature_evidence"
    ].iterrows():

        feature = row[
            "feature"
        ]

        components = ", ".join(
            AREA_DETAILS[
                feature
            ]["components"]
        )

        area_blocks.append(
            f"""
Area: {row['label']}
Construct Score: {row['score']:.2f}/5
Interpretation: {row['level']}
SHAP contribution: {row['shap_value']:.4f}
Model effect: {row['effect']}
Components: {components}
Career relevance: {row['career_relevance']}
"""
        )

    evidence_text = "\n".join(
        area_blocks
    )

    item_lines = []

    for _, row in (
        item_evidence.iterrows()
    ):

        item_lines.append(
            (
                f"- {row['area']} | "
                f"{row['skill']}: "
                f"{row['score']:.1f}/5 "
                f"({row['level']})"
            )
        )

    item_text = "\n".join(
        item_lines
    )

    resource_text = json.dumps(
        selected_resources,
        indent=2
    )

    prompt = f"""
Analyze the competence evidence below and generate a personalized
career-development recommendation.

The Employability prediction has already been generated by a trained
machine learning model.

Predicted Employability Score:
{evidence['predicted_evs_avg']:.3f} / 5

Construct-Level Model Evidence:
{evidence_text}

Individual Survey Evidence:
{item_text}

Strongest observed competence construct:
{evidence['strongest_feature']['label']}

Lowest observed competence construct:
{evidence['weakest_feature']['label']}

Approved Learning Resources:
{resource_text}

For each competence area:

1. Identify specific strengths using the individual survey item scores.
2. Identify specific areas that may benefit from further development.
3. Explain briefly why the competence area is relevant to career readiness.
4. Provide 2-3 practical and realistic improvement suggestions.

Then provide:

- Top 3 Strengths
- Top 3 Areas for Improvement
- Personalized Action Plan
- Overall Career Readiness Summary
- Recommended Next Steps
- Recommended Learning Resources

Important rules:

- Do not calculate or modify the Employability prediction.
- Use construct-level SHAP evidence only to explain model behaviour.
- Use individual survey item scores to identify specific strengths and weaknesses.
- Do not infer a specific skill strength or weakness from a construct average alone.
- Use only the supplied evidence.
- Use learning resources only from the Approved Learning Resources list.
- Do not invent URLs, qualifications, experience, or personal facts.
- Do not make causal claims.
- Do not make medical or psychological diagnoses.
- Do not assume information not contained in the evidence.
- Keep recommendations supportive, practical, and specific.
- Do not state or imply that improving a competence will necessarily increase Employability.
- When discussing SHAP, describe only how the feature influenced the model prediction.
- Recommendations should be framed as career-development opportunities, not guaranteed methods for increasing the predicted score.
"""

    return prompt

In [197]:
recommendation_schema = {

    "type": "object",

    "properties": {

        "area_analysis": {

            "type": "array",

            "items": {

                "type": "object",

                "properties": {

                    "area": {
                        "type": "string"
                    },

                    "strengths": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },

                    "areas_for_improvement": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },

                    "career_readiness_impact": {
                        "type": "string"
                    },

                    "practical_suggestions": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },

                "required": [
                    "area",
                    "strengths",
                    "areas_for_improvement",
                    "career_readiness_impact",
                    "practical_suggestions"
                ],

                "additionalProperties": False
            }
        },

        "top_3_strengths": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "top_3_areas_for_improvement": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "personalized_action_plan": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "overall_career_readiness_summary": {
            "type": "string"
        },

        "recommended_next_steps": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },

        "recommended_learning_resources": {

            "type": "array",

            "items": {

                "type": "object",

                "properties": {

                    "area": {
                        "type": "string"
                    },

                    "provider": {
                        "type": "string"
                    },

                    "resource_type": {
                        "type": "string"
                    },

                    "topic": {
                        "type": "string"
                    },

                    "reason": {
                        "type": "string"
                    }
                },

                "required": [
                    "area",
                    "provider",
                    "resource_type",
                    "topic",
                    "reason"
                ],

                "additionalProperties": False
            }
        }
    },

    "required": [
        "area_analysis",
        "top_3_strengths",
        "top_3_areas_for_improvement",
        "personalized_action_plan",
        "overall_career_readiness_summary",
        "recommended_next_steps",
        "recommended_learning_resources"
    ],

    "additionalProperties": False
}

In [198]:
OPENAI_MODEL = "gpt-5-mini"

In [199]:
def generate_api_recommendation(
    evidence,
    item_evidence,
    selected_resources
):

    prompt = build_api_prompt(
        evidence,
        item_evidence,
        selected_resources
    )

    response = client.responses.create(

        model=OPENAI_MODEL,

        input=[
            {
                "role": "system",
                "content": (
                    "You generate evidence-grounded "
                    "career-development guidance."
                )
            },

            {
                "role": "user",
                "content": prompt
            }
        ],

        text={
            "format": {
                "type": "json_schema",
                "name": (
                    "career_recommendation"
                ),
                "strict": True,
                "schema": (
                    recommendation_schema
                )
            }
        }
    )

    result = json.loads(
        response.output_text
    )

    result[
        "source"
    ] = "openai_api"

    return result

In [200]:
def generate_recommendation(
    evidence,
    item_evidence
):

    selected_resources = (
        select_learning_resources(
            evidence
        )
    )

    try:

        result = (
            generate_api_recommendation(
                evidence,
                item_evidence,
                selected_resources
            )
        )

        return result

    except Exception as error:

        print(
            "API unavailable or failed."
        )

        print(
            "Using rule-based fallback."
        )

        print(
            "API error:",
            error
        )

        return (
            generate_rule_based_recommendation(
                evidence,
                item_evidence
            )
        )

In [201]:
recommendation = (
    generate_recommendation(
        evidence,
        item_evidence
    )
)

print(
    "Recommendation source:",
    recommendation[
        "source"
    ]
)

print(
    "Available keys:",
    recommendation.keys()
)

Recommendation source: openai_api
Available keys: dict_keys(['area_analysis', 'top_3_strengths', 'top_3_areas_for_improvement', 'personalized_action_plan', 'overall_career_readiness_summary', 'recommended_next_steps', 'recommended_learning_resources', 'source'])


In [202]:
print(
    "Predicted Employability Score:",
    round(
        evidence[
            "predicted_evs_avg"
        ],
        3
    )
)

print(
    "\n=============================="
)

print(
    "AREA-BY-AREA ANALYSIS"
)

print(
    "=============================="
)

for area in recommendation[
    "area_analysis"
]:

    print(
        f"\n{area['area']}"
    )

    print(
        "\nStrengths:"
    )

    for item in area[
        "strengths"
    ]:
        print(
            "-",
            item
        )

    print(
        "\nAreas for Improvement:"
    )

    for item in area[
        "areas_for_improvement"
    ]:
        print(
            "-",
            item
        )

    print(
        "\nWhy it matters:"
    )

    print(
        area[
            "career_readiness_impact"
        ]
    )

    print(
        "\nPractical Suggestions:"
    )

    for item in area[
        "practical_suggestions"
    ]:
        print(
            "-",
            item
        )


print(
    "\n=============================="
)

print(
    "TOP 3 STRENGTHS"
)

print(
    "=============================="
)

for item in recommendation[
    "top_3_strengths"
]:
    print(
        "-",
        item
    )


print(
    "\n=============================="
)

print(
    "TOP 3 AREAS FOR IMPROVEMENT"
)

print(
    "=============================="
)

for item in recommendation[
    "top_3_areas_for_improvement"
]:
    print(
        "-",
        item
    )


print(
    "\n=============================="
)

print(
    "PERSONALIZED ACTION PLAN"
)

print(
    "=============================="
)

for item in recommendation[
    "personalized_action_plan"
]:
    print(
        "-",
        item
    )


print(
    "\n=============================="
)

print(
    "CAREER READINESS SUMMARY"
)

print(
    "=============================="
)

print(
    recommendation[
        "overall_career_readiness_summary"
    ]
)


print(
    "\n=============================="
)

print(
    "RECOMMENDED NEXT STEPS"
)

print(
    "=============================="
)

for item in recommendation[
    "recommended_next_steps"
]:
    print(
        "-",
        item
    )


print(
    "\n=============================="
)

print(
    "LEARNING RESOURCES"
)

print(
    "=============================="
)

for resource in recommendation[
    "recommended_learning_resources"
]:

    print(
        f"\n{resource['area']}"
    )

    print(
        "Provider:",
        resource[
            "provider"
        ]
    )

    print(
        "Type:",
        resource[
            "resource_type"
        ]
    )

    print(
        "Topic:",
        resource[
            "topic"
        ]
    )

    print(
        "Why:",
        resource[
            "reason"
        ]
    )


print(
    "\nRecommendation Source:",
    recommendation[
        "source"
    ]
)

Predicted Employability Score: 4.055

AREA-BY-AREA ANALYSIS

Life & Career Skills

Strengths:
- Adaptability to changing situations (4.0/5)
- Communication with diverse team members (4.0/5)
- Leadership ability (4.0/5)

Areas for Improvement:
- Project planning and execution (3.0/5)

Why it matters:
Life & Career Skills support working effectively with others, adjusting to changing conditions, and taking on responsibility. The model-level SHAP contribution for this construct was -0.0088, indicating a small negative influence on the predicted employability score.

Practical Suggestions:
- Complete a short introduction to project management (e.g., watch a 'Project Management Fundamentals' video) and apply one basic framework (define scope, set milestones, assign tasks) to a small project.
- Use simple planning tools (checklist, calendar, Kanban board) for a 4–8 week task to practice breaking work into phases and tracking progress.
- Volunteer to manage a small team task (even informal) a

In [203]:
final_output = {

    "prediction": {
        "EVS_avg": round(
            evidence[
                "predicted_evs_avg"
            ],
            4
        )
    },

    "competence_scores": {
        row["feature"]:
        row["score"]

        for _, row in evidence[
            "feature_evidence"
        ].iterrows()
    },

    "item_level_evidence": [

        {
            "item": row["item"],
            "skill": row["skill"],
            "area": row["area"],
            "score": row["score"],
            "level": row["level"]
        }

        for _, row in (
            item_evidence.iterrows()
        )
    ],

    "shap_explanation": {

        row["feature"]:
        round(
            row["shap_value"],
            6
        )

        for _, row in evidence[
            "feature_evidence"
        ].iterrows()
    },

    "area_analysis": (
        recommendation[
            "area_analysis"
        ]
    ),

    "top_3_strengths": (
        recommendation[
            "top_3_strengths"
        ]
    ),

    "top_3_areas_for_improvement": (
        recommendation[
            "top_3_areas_for_improvement"
        ]
    ),

    "personalized_action_plan": (
        recommendation[
            "personalized_action_plan"
        ]
    ),

    "overall_career_readiness_summary": (
        recommendation[
            "overall_career_readiness_summary"
        ]
    ),

    "recommended_next_steps": (
        recommendation[
            "recommended_next_steps"
        ]
    ),

    "recommended_learning_resources": (
        recommendation[
            "recommended_learning_resources"
        ]
    ),

    "recommendation_source": (
        recommendation[
            "source"
        ]
    )
}

final_output

{'prediction': {'EVS_avg': 4.0548},
 'competence_scores': {'LifeCareer_avg': 3.75,
  'LearningInnovation_avg': 3.3333333333333335,
  'ITMedia_avg': 4.333333333333333},
 'item_level_evidence': [{'item': 'C1',
   'skill': 'Adaptability to changing situations',
   'area': 'Life & Career Skills',
   'score': 4.0,
   'level': 'Strong'},
  {'item': 'C2',
   'skill': 'Communication with diverse team members',
   'area': 'Life & Career Skills',
   'score': 4.0,
   'level': 'Strong'},
  {'item': 'C3',
   'skill': 'Project planning and execution',
   'area': 'Life & Career Skills',
   'score': 3.0,
   'level': 'Developing'},
  {'item': 'C4',
   'skill': 'Leadership ability',
   'area': 'Life & Career Skills',
   'score': 4.0,
   'level': 'Strong'},
  {'item': 'C5',
   'skill': 'Reasoning and decision-making',
   'area': 'Learning & Innovation Skills',
   'score': 3.0,
   'level': 'Developing'},
  {'item': 'C6',
   'skill': 'Communication and collaboration',
   'area': 'Learning & Innovation Skil

In [204]:
OUTPUT_PATH = (
    RESULT_DIR
    / "final_sample_recommendation.json"
)

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_output,
        file,
        indent=4,
        ensure_ascii=False
    )

print(
    "Recommendation saved:",
    OUTPUT_PATH
)

Recommendation saved: d:\internship-employability-project\reports\phase2\recommendation_system\final_sample_recommendation.json


In [205]:
test_cases = [

    {
        "name": "High Competence Profile",

        "survey": {
            "C1": 5,
            "C2": 4,
            "C3": 4,
            "C4": 5,

            "C5": 4,
            "C6": 5,
            "C7": 4,

            "C8": 4,
            "C9": 5,
            "C10": 4
        }
    },

    {
        "name": "Mixed Competence Profile",

        "survey": {
            "C1": 4,
            "C2": 4,
            "C3": 3,
            "C4": 3,

            "C5": 3,
            "C6": 4,
            "C7": 3,

            "C8": 4,
            "C9": 5,
            "C10": 4
        }
    },

    {
        "name": "Development Needed Profile",

        "survey": {
            "C1": 3,
            "C2": 3,
            "C3": 2,
            "C4": 3,

            "C5": 2,
            "C6": 3,
            "C7": 2,

            "C8": 3,
            "C9": 3,
            "C10": 2
        }
    }
]

In [206]:
validation_results = []

for case in test_cases:

    survey = case[
        "survey"
    ]

    case_input = {

        "LifeCareer_avg": np.mean([
            survey["C1"],
            survey["C2"],
            survey["C3"],
            survey["C4"]
        ]),

        "LearningInnovation_avg": np.mean([
            survey["C5"],
            survey["C6"],
            survey["C7"]
        ]),

        "ITMedia_avg": np.mean([
            survey["C8"],
            survey["C9"],
            survey["C10"]
        ])
    }

    case_evidence = (
        build_recommendation_evidence(
            case_input,
            selected_model,
            explainer,
            scaler
        )
    )

    case_item_evidence = (
        build_item_evidence(
            survey
        )
    )

    case_recommendation = (
        generate_recommendation(
            case_evidence,
            case_item_evidence
        )
    )

    validation_results.append({

        "profile": case[
            "name"
        ],

        "predicted_evs_avg": round(
            case_evidence[
                "predicted_evs_avg"
            ],
            4
        ),

        "strongest_construct": (
            case_evidence[
                "strongest_feature"
            ]["label"]
        ),

        "weakest_construct": (
            case_evidence[
                "weakest_feature"
            ]["label"]
        ),

        "top_strengths": (
            case_recommendation[
                "top_3_strengths"
            ]
        ),

        "improvement_areas": (
            case_recommendation[
                "top_3_areas_for_improvement"
            ]
        ),

        "source": (
            case_recommendation[
                "source"
            ]
        )
    })

validation_results

[{'profile': 'High Competence Profile',
  'predicted_evs_avg': 4.4647,
  'strongest_construct': 'Life & Career Skills',
  'weakest_construct': 'Learning & Innovation Skills',
  'top_strengths': ['Adaptability to changing situations (5.0/5)',
   'Leadership ability (5.0/5)',
   'Communication and collaboration (5.0/5)'],
  'improvement_areas': ['Project planning and execution (4.0/5)',
   'Creativity and innovation (4.0/5)',
   'Information access and evaluation (4.0/5)'],
  'source': 'openai_api'},
 {'profile': 'Mixed Competence Profile',
  'predicted_evs_avg': 3.9997,
  'strongest_construct': 'IT & Media Skills',
  'weakest_construct': 'Learning & Innovation Skills',
  'top_strengths': ['Media communication skills (5.0/5)',
   'Information access and evaluation (4.0/5)',
   'Communication with diverse team members (4.0/5)'],
  'improvement_areas': ['Project planning and execution (3.0/5)',
   'Leadership ability (3.0/5)',
   'Reasoning and decision-making (3.0/5)'],
  'source': 'opena

In [207]:
with open(
    RESULT_DIR
    / "recommendation_validation_results.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        validation_results,
        file,
        indent=4,
        ensure_ascii=False
    )

print(
    "Recommendation validation saved."
)

Recommendation validation saved.


## Step 10 Conclusion

A personalized and evidence-grounded career recommendation system was developed using the selected Competence → Employability model and SHAP explanations.

The system evaluates Life & Career Skills, Learning & Innovation Skills, and IT & Media Skills and generates area-specific strengths, improvement needs, career-readiness explanations, practical suggestions, an action plan, and recommended next steps.

The OpenAI API is used only to transform validated model evidence into clear natural-language guidance. It does not calculate or modify the Employability prediction.

Learning resources are selected from a predefined resource library to reduce the risk of unsupported or fabricated recommendations. A rule-based fallback mechanism is also provided so that guidance remains available when the external API is unavailable.

The resulting recommendation structure is suitable for integration with the final Employability prediction system.